In [ ]:
# ----------------------------
# Training MobileNetV3 Small (CPU Optimized)
# ----------------------------

import os
os.environ["OMP_NUM_THREADS"] = "4"

# =========================================================
# System & Performance Settings
# =========================================================
import torch
torch.set_num_threads(4)
# torch.set_num_interop_threads(2)

import datetime
from tqdm import tqdm
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

# =========================================================
# Dataset Paths
# =========================================================
train_dir = "model/dataset-splitted/train"
val_dir = "model/dataset-splitted/val"

# =========================================================
# Data Transforms (smaller image size for speed)
# =========================================================
data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((128, 128)),  # smaller size = faster
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    "val": transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

# =========================================================
# Datasets & Dataloaders
# =========================================================
train_data = datasets.ImageFolder(train_dir, transform=data_transforms["train"])
val_data   = datasets.ImageFolder(val_dir, transform=data_transforms["val"])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=2)

# =========================================================
# Model Setup (MobileNetV3 Small)
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
# Replace last classifier layer
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(train_data.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# =========================================================
# Folder for saving models
# =========================================================
os.makedirs("final-model-test-2", exist_ok=True)

# =========================================================
# Training Loop
# =========================================================
best_acc = 0
epochs = 32

for epoch in range(epochs):
    model.train()
    train_loss, correct, total = 0, 0, 0

    print(f"\nEpoch {epoch+1}/{epochs}")
    for imgs, labels in tqdm(train_loader, desc="Training"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total

    # ----------------------------
    # Validation
    # ----------------------------
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc="Validation"):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100 * val_correct / val_total
    print(f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    # ----------------------------
    # Save best model
    # ----------------------------
    if val_acc > best_acc:
        best_acc = val_acc
        timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        save_path = f"final-model-test-2/dress-type-model-{timestamp}.pth"
        torch.save(model.state_dict(), save_path)
        print(f"Saved best model: {save_path}")

print(f"\nTraining complete! Best validation accuracy: {best_acc:.2f}%")

In [ ]:
# ----------------------------
# Training mobilenet_v3_small
# ----------------------------

# Data transforms (augmentation + normalization)
data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    "val": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

# Datasets and loaders
train_data = datasets.ImageFolder(train_dir, transform=data_transforms["train"])
val_data   = datasets.ImageFolder(val_dir, transform=data_transforms["val"])

torch.set_num_threads(4)  # or number of CPU cores
torch.set_num_interop_threads(2)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=2)

# Model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, len(train_data.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Training loop
best_acc = 0
for epoch in range(10):  # increase epochs for better accuracy
    model.train()
    train_loss, correct, total = 0, 0, 0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/10"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100 * val_correct / val_total
    print(f"Epoch {epoch+1}: Train Acc = {train_acc:.2f}%, Val Acc = {val_acc:.2f}%")

    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        torch.save(model.state_dict(), f"final-model-test-2/dress-type-model-{timestamp}.pth")

print(f"Training complete! Best validation accuracy: {best_acc:.2f}%")

In [ ]:
# ----------------------------
# Training MobileNetV3 Small (CPU Optimized)
# ----------------------------

import os
os.environ["OMP_NUM_THREADS"] = "4"

import torch
torch.set_num_threads(4)
torch.set_num_interop_threads(2)

import datetime
from tqdm import tqdm
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader


In [ ]:
# 1️⃣ Load your saved model

import torch
from torchvision import models
import torch.nn as nn

# Load the model architecture
model = models.resnet50(weights=None)  # or weights='IMAGENET1K_V2' if you want pretrained
model.fc = nn.Linear(model.fc.in_features, num_classes)  # make sure num_classes match

# Load previously saved weights
model.load_state_dict(torch.load("path/to/your/dress_type_model_last.pth"))


In [ ]:
2️⃣ Prepare your new dataset

from torchvision import datasets, transforms

data_transforms = {
    "train": transforms.Compose([...]),  # same as your previous training transforms
    "val": transforms.Compose([...])
}

train_data = datasets.ImageFolder("path/to/new/train", transform=data_transforms["train"])
val_data = datasets.ImageFolder("path/to/new/val", transform=data_transforms["val"])


In [ ]:
3️⃣ Continue training

import torch.optim as optim
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Fine-tune for more epochs
for epoch in range(additional_epochs):
    model.train()
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
